In [ ]:
# !pip install tensorflow pillow mathplotlib

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet import preprocess_input
import pathlib

# 1. Dataset path
data_dir = pathlib.Path(r"C:\Users\turzo\Documents\Neural\flower_photos")

# 2. Load train and validation data
train_data = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

val_data = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

class_names = train_data.class_names
print("Classes:", class_names)

# 3. Pretrained ResNet-50
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# 4. Fine-tuning
base_model.trainable = True

for layer in base_model.layers[:-10]:
    layer.trainable = False

# 5. Final model
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(5, activation="softmax")
])

# 6. Compile and train
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_data,
    validation_data=val_data,
    epochs=2
)

In [ ]:
loss, acc = model.evaluate(val_data)
print("Test Accuracy:", acc)